# 01: Corpus Preparation and Retriever Setup

Follow `01_corpus_and_retrievers.md` alongside this notebook.

Run each cell group, inspect the output, then move to the next.

## Cell group 1: Imports and environment check

In [1]:
import json
from collections import Counter
from pathlib import Path
import sys

sys.path.insert(0, str(Path("scripts").resolve()))

import spacy
from rank_bm25 import BM25Okapi
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

from retriever import tokenise

print("All imports OK")

All imports OK


In [ ]:
from log_setup import setup_logging

logger = setup_logging("01_corpus_and_retrievers")

In [3]:
# verify spaCy model is available
nlp = spacy.load('en_core_web_sm')
print(f'spaCy model loaded: {nlp.meta["name"]} v{nlp.meta["version"]}')

spaCy model loaded: core_web_sm v3.8.0


In [4]:
# verify sentence-transformers can load the embedding model
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
test_embed = embeddings.embed_query('test')
print(f'Embedding model OK, vector dimension: {len(test_embed)}')

05:32:27  INFO      No device provided, using cuda:0


05:32:27  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


05:32:27  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"


05:32:27  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


05:32:27  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"


...

05:32:30  INFO      HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


05:32:30  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"


05:32:30  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


05:32:30  INFO      HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"


Embedding model OK — vector dimension: 384


## Cell group 2: Load corpus

The 2,568 clauses were pre-parsed from six regulatory PDFs (MLR 2017, POCA 2002,
JMLSG Parts I & II, FCA FCG, FATF 40) by `scripts/corpus.py` and saved to
`data/clauses.jsonl`. Parsing is done once; the JSONL file is the source of truth
for all downstream steps.

In [5]:
all_clauses = [
    json.loads(line)
    for line in Path("data/clauses.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

counts = Counter(c["source"] for c in all_clauses)
print(f"Total clauses: {len(all_clauses)}\n")
for src_name, n in counts.most_common():
    print(f"  {src_name:<20} {n:>5}")

Total clauses: 2568

  jmlsg_2                901
  poca_2002              764
  jmlsg_1                567
  mlr_2017               218
  fatf_40                 69
  fca_fcg                 49


## Cell group 3: Dense retriever (ChromaDB)

In [6]:
CHROMA_DIR = "data/chroma_db"
clauses_for_index = all_clauses

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

chroma_path = Path(CHROMA_DIR)
if chroma_path.exists() and any(chroma_path.iterdir()):
    logger.info("Loading persisted ChromaDB from %s", CHROMA_DIR)
    vectorstore = Chroma(
        persist_directory=CHROMA_DIR,
        embedding_function=embeddings,
        collection_name="aml_clauses",
    )
    logger.info("Collection has %d documents", vectorstore._collection.count())
else:
    logger.warning("ChromaDB not found at %s: run scripts/build_chroma.py first", CHROMA_DIR)
    vectorstore = None

05:32:31  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


05:32:31  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"


05:32:31  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


05:32:31  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"


05:32:31  INFO      Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.


...

05:32:33  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"


05:32:33  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


05:32:33  INFO      HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"


05:32:33  INFO      Loading persisted ChromaDB from data/chroma_db


05:32:34  INFO      Collection has 2568 documents


## Cell group 4: Sparse retriever (BM25)

In [7]:
logger.info("Building BM25 index over %d clauses...", len(clauses_for_index))
corpus_tokens = [tokenise(c["text"]) for c in clauses_for_index]
bm25 = BM25Okapi(corpus_tokens)
logger.info("BM25 built, vocab: %d terms", len(bm25.idf))

print("Tokens for 'customer due diligence requirements':", tokenise("customer due diligence requirements"))

05:32:34  INFO      Building BM25 index over 2568 clauses...


05:32:34  INFO      BM25 built — vocab: 7454 terms


Tokens for 'customer due diligence requirements': ['customer', 'due', 'diligence', 'requirements']


## Cell group 5: Smoke test

In [8]:
TEST_QUERY = "What are the customer due diligence requirements under MLR 2017?"

# --- Dense retriever ---
if vectorstore is not None:
    dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    dense_results = dense_retriever.invoke(TEST_QUERY)
    print("=== Dense results ===")
    for r in dense_results:
        print(f"  {r.metadata['clause_id']:<35} {r.page_content[:80]}")
else:
    print("Dense retriever not available: run build_chroma.py first")

# --- BM25 retriever ---
query_tokens = tokenise(TEST_QUERY)
scores = bm25.get_scores(query_tokens)
top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]

print("\n=== BM25 results ===")
for i in top_idx:
    c = clauses_for_index[i]
    print(f"  {c['clause_id']:<35} score={scores[i]:.2f}  {c['text'][:80]}")


=== Dense results ===
  jmlsg_1_5.1.1                       5.1.1 The ML Regulations 2017 (as amended) specify CDD measures that are
require
  jmlsg_2_9.16                        9.16 When dealing with other types of customers, firms would normally be able to
  jmlsg_1_5.1                         5.1 Meaning of customer due diligence measures and ongoing monitoring
  jmlsg_2_18.93                       18.93 Where business does not fall within the scope of the ML Regulations, e.g.,
  mlr_2017_sch6_para7                 7. The requirements specified in this paragraph are those imposed in—
(a) regula

=== BM25 results ===
  jmlsg_2_18.93                       score=13.73  18.93 Where business does not fall within the scope of the ML Regulations, e.g.,
  jmlsg_2_7.1                         score=13.61  7.1 This sectoral guidance helps firms to interpret how the risk-based approach 
  mlr_2017_sch6_para7                 score=12.91  7. The requirements specified in this paragraph are those